# 1. Introducción

**Problema industrial:** Integración entre modelos de ingeniería en Excel y análisis en Python.

**Activo analizado:** PUMP101 — modelo híbrido Excel + Python.

**Origen de datos:** Mediciones PI y parámetros de diseño en `modelo_ingenieria.xlsx`.

**Objetivo del análisis:** Calcular KPIs de eficiencia y comparar medición vs. modelo.


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Cálculo de KPIs: eficiencia volumétrica y desviación vs. diseño.

In [ ]:
umbrales = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Umbrales")
modelo = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Modelo")

wide = df_good.pivot_table(index="Timestamp", columns="Tag", values="Value", aggfunc="mean")
caudal_medido = wide["PUMP101.FLOW_RATE"].mean()
caudal_diseno = float(modelo.loc[modelo["Parametro"] == "Caudal_Diseno", "Valor"].iloc[0])
eficiencia = (caudal_medido / caudal_diseno) * 100

resultados_export = pd.DataFrame({
    "KPI": ["Caudal_Medido", "Caudal_Diseno", "Eficiencia_%"],
    "Valor": [caudal_medido, caudal_diseno, eficiencia],
})
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
wide["PUMP101.FLOW_RATE"].resample("1D").mean().plot(ax=ax, label="Medido", color="steelblue")
ax.axhline(caudal_diseno, color="red", linestyle="--", label=f"Diseño={caudal_diseno:.0f} m3/h")
ax.set_title("Caudal medido vs. diseño")
ax.set_ylabel("m3/h")
ax.legend()
ax.grid(True, alpha=0.3)


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="KPIs", index=False)
    modelo.to_excel(writer, sheet_name="Modelo", index=False)
    umbrales.to_excel(writer, sheet_name="Umbrales", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

La eficiencia por debajo del 100% sugiere revisar desgaste de impulsor o restricciones en succión. Actualizar el modelo Excel con los nuevos coeficientes calibrados desde PI.
